In [2]:
!pip install dotenv

  Using cached python_dotenv-1.2.4-py3-none-any.whl.metadata (29 kB)
Using cached python_dotenv-1.2.4-py3-none-any.whl (23 kB)

   ---------------------------------------- 0/2 [python-dotenv]
   ---------------------------------------- 0/2 [python-dotenv]
   ---------------------------------------- 0/2 [python-dotenv]
   ---------------------------------------- 0/2 [python-dotenv]
   -------------------- ------------------- 1/2 [dotenv]
   -------------------- ------------------- 1/2 [dotenv]
   -------------------- ------------------- 1/2 [dotenv]
   -------------------- ------------------- 1/2 [dotenv]
   ---------------------------------------- 2/2 [dotenv]



In [4]:
!pip install streamlit langchain chromadb google-generativeai fastapi uvicorn

  Using cached streamlit-1.66.0-py3-none-any.whl.metadata (10 kB)
  Using cached langchain-1.4.6-py3-none-any.whl.metadata (6.2 kB)
  Using cached chromadb-1.5.9-cp39-abi3-win_amd64.whl.metadata (5.1 kB)
  Using cached fastapi-0.153.0-py3-none-any.whl.metadata (26 kB)
  Using cached uvicorn-0.54.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached altair-6.3.0-py3-none-any.whl.metadata (11 kB)
  Using cached pandas-3.0.6-cp314-cp314-win_amd64.whl.metadata (19 kB)
  Using cached pydeck-0.9.3-py2.py3-none-any.whl.metadata (4.2 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached starlette-1.9.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached anyio-4.15.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached langchain_core-1.6.12-py3-none-any.whl.metadata (4.8 kB)
  Using cached langgraph-1.2.16-py3-none-any.whl.metadata (4.9 kB)
 

In [3]:
import os
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings

load_dotenv()

# Set up vector store & embeddings
persist_dir = os.path.abspath('./Models/chroma_db')
embeddings = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')
vector_store = Chroma(
    persist_directory=persist_dir, embedding_function=embeddings
)
retriever = vector_store.as_retriever(search_kwargs={'k': 3})

# Initialize model
llm = ChatGoogleGenerativeAI(model='gemini-3.8-flash', temperature=0.2)

# Prompt
template = """You are an expert legal advisor for entrepreneurs in Egypt.
Context:
{context}

Question: {question}

Answer:"""

prompt = ChatPromptTemplate.from_template(template)


def format_docs(docs):
  return '\n\n'.join(doc.page_content for doc in docs)


chain = (
    {
        'context': retriever | format_docs,
        'question': RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

# Test query
query = 'How do I register an LLC via GAFI?'
print(chain.invoke(query))

ModuleNotFoundError: No module named 'langchain_chroma'